# ENDES 2025: anemia infantil en la macrorregión norte

Este notebook consolida, limpia y transforma los módulos ENDES 2025 disponibles en `BASE DE DATOS` para preparar una matriz de entrada para una red MLP. Recorre las subcarpetas, valida cada variable solicitada contra el diccionario Markdown asociado y lee únicamente esas columnas.

**Llaves:** antropometría y salud infantil por `CASEID` + orden del niño (`HWIDX`/`HIDX`/`IDX95`); madre por `CASEID`; hogar por `HHID` derivado del primer componente de `CASEID`. Se verifican cardinalidades para impedir que un merge multiplique registros.

**Dependencias:** `pandas`, `numpy` y `scikit-learn` (`pip install pandas numpy scikit-learn`).

**Importante sobre el objetivo:** `REC44` aporta hemoglobina infantil cruda (`HW53`), no ajustada ni categorizada. Se deriva la hemoglobina ajustada y la clase de anemia aquí, usando una corrección convencional por altitud y umbrales de 2024 configurables. Confirme ambos con el protocolo clínico aprobado para el estudio antes de publicar resultados. No se usa hemoglobina en las features ni se exporta como predictor.

In [10]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from IPython.display import Markdown


def show_table(title, frame):
    display(Markdown(f"#### {title}"))
    display(frame)

# Ejecución esperada desde C:\PROYECTODSI o desde la carpeta que contiene BASE DE DATOS.
candidates = [Path.cwd() / "BASE DE DATOS", Path.cwd().parent / "BASE DE DATOS"]
BASE_DIR = next((p.resolve() for p in candidates if p.is_dir()), None)
if BASE_DIR is None:
    raise FileNotFoundError(
        "No se encontró 'BASE DE DATOS'. Abra el notebook desde C:\\PROYECTODSI "
        "o actualice BASE_DIR a la ruta de los datos."
    )
OUTPUT_CSV = BASE_DIR.parent / "ENDES_2025_MLP_macrorregion_norte.csv"

# Mapa variable ENDES -> nombre claro. La selección se contrasta con CSV y diccionario.
SOURCE_MAP = {
    "REC44": {
        "CASEID": "caseid", "HWIDX": "child_idx", "HW1": "age_months",
        "HW2": "weight_raw", "HW3": "height_raw", "HW5": "haz_raw",
        "HW8": "waz_raw", "HW11": "whz_raw", "HW53": "hb_raw",
    },
    "RECH23": {
        "HHID": "hhid", "HV201": "water_source", "HV205": "sanitation",
        "HV213": "floor_material", "HV214": "wall_material",
        "HV215": "roof_material", "HV270": "wealth_quintile",
        "HV271": "wealth_score",
    },
    "RECH0": {
        "HHID": "hhid", "HV009": "household_persons",
        "HV014": "children_under5", "HV024": "department_code",
        "HV025": "residence_code", "HV040": "altitude_m",
    },
    "RECH1": {
        "HHID": "hhid", "HVIDX": "roster_idx", "HV104": "sex_code",
        "HV105": "age_years", "HV112": "mother_line",
    },
    "REC0111": {
        "CASEID": "caseid", "HHID": "hhid_mother",
        "V106": "mother_education", "V131": "mother_language",
    },
    "REC91": {"CASEID": "caseid", "S484": "juntos_code"},
    "REC43": {
        "CASEID": "caseid", "HIDX": "child_idx",
        "H11": "diarrhea_code", "H22": "fever_code",
        "H31": "cough_code", "H34": "vitamin_a_6m_code",
    },
    "REC95": {
        "CASEID": "caseid", "IDX95": "child_idx",
        "S465DB_A": "iron_syrup_code", "S465DB_B": "iron_drops_code",
        "S465DB_C": "iron_micronutrients_code",
    },
}

DEPARTMENT_CODES = {
    1: "Amazonas", 2: "Áncash", 6: "Cajamarca", 13: "La Libertad",
    14: "Lambayeque", 20: "Piura", 22: "San Martín", 24: "Tumbes",
}
TARGET_ORDER = ["sin_anemia", "anemia_leve", "anemia_moderada", "anemia_severa"]
AGE_MIN_MONTHS, AGE_MAX_MONTHS = 6, 59

# Categorías vigentes de referencia (2024); verificar antes de uso clínico.
HB_CUTOFF_6_23_MONTHS = 10.5
HB_CUTOFF_24_59_MONTHS = 11.0
HB_SEVERE_CUTOFF = 7.0
HB_MODERATE_CUTOFF = 10.0
ALTITUDE_MIN_METERS = 1000

## Descubrimiento y lectura guiada por diccionarios

Se localizan CSV y diccionarios por código de módulo en todas las subcarpetas. Si falta un módulo, una columna requerida o su entrada en el diccionario, el notebook se detiene con un error descriptivo; no sustituye variables por aproximación.

In [11]:
csv_paths = sorted(BASE_DIR.rglob("*_2025.csv"))
dictionary_paths = sorted(BASE_DIR.rglob("Diccionario - *.md"))
if not csv_paths:
    raise FileNotFoundError(f"No se encontraron CSV *_2025.csv dentro de {BASE_DIR}")

csv_by_code = {}
for path in csv_paths:
    code_name = path.stem.rsplit("_", 1)[0]
    if code_name in csv_by_code:
        raise ValueError(f"Hay más de un CSV para el módulo {code_name}: {csv_by_code[code_name]}, {path}")
    csv_by_code[code_name] = path

dictionary_by_code = {}
for path in dictionary_paths:
    match = re.fullmatch(r"Diccionario - ([A-Za-z0-9]+)\.md", path.name)
    if match:
        dictionary_by_code[match.group(1).upper()] = path

discovered = []
for csv_path in csv_paths:
    module = csv_path.stem.rsplit("_", 1)[0].upper()
    dictionary = dictionary_by_code.get(module)
    discovered.append({
        "módulo": module,
        "carpeta": csv_path.parent.name,
        "CSV": csv_path.name,
        "diccionario": dictionary.name if dictionary else "FALTA",
        "usado": module in SOURCE_MAP,
    })
show_table("Inventario de módulos CSV y diccionarios", pd.DataFrame(discovered))

missing_modules = sorted(set(SOURCE_MAP) - set(csv_by_code))
if missing_modules:
    raise FileNotFoundError(f"Faltan CSV requeridos para estos módulos: {missing_modules}")

def dictionary_has_code(dictionary_text, variable):
    return re.search(
        rf"(?<![A-Z0-9_]){re.escape(variable.upper())}(?![A-Z0-9_])",
        dictionary_text.upper(),
    ) is not None

def load_selected_module(module):
    csv_path = csv_by_code[module]
    dictionary_path = dictionary_by_code.get(module)
    if dictionary_path is None:
        raise FileNotFoundError(f"No existe Diccionario - {module}.md para {csv_path}")
    dictionary_text = dictionary_path.read_text(encoding="utf-8-sig", errors="replace")
    header = pd.read_csv(csv_path, encoding="utf-8-sig", nrows=0).columns.tolist()
    mapping = SOURCE_MAP[module]
    required_codes = set(mapping)
    absent_dictionary = sorted(v for v in required_codes if not dictionary_has_code(dictionary_text, v))
    absent_csv = sorted(required_codes - set(header))
    if absent_dictionary or absent_csv:
        raise ValueError(
            f"{module}: variables faltantes. Diccionario={absent_dictionary}; CSV={absent_csv}"
        )
    selected = list(mapping)
    frame = pd.read_csv(
        csv_path, encoding="utf-8-sig", usecols=selected, dtype=str,
        keep_default_na=False, low_memory=False,
    ).rename(columns=mapping)
    for key in ("caseid", "hhid", "hhid_mother"):
        if key in frame:
            frame[key] = frame[key].astype("string").str.strip()
    frame = frame.replace(r"^\s*$", pd.NA, regex=True)
    return frame

sources = {module: load_selected_module(module) for module in SOURCE_MAP}
print("• Variables seleccionadas por módulo (validadas en su diccionario):")
for module, frame in sources.items():
    print(f"  • {module}: {list(frame.columns)}")


#### Inventario de módulos CSV y diccionarios

,módulo,carpeta,CSV,diccionario,usado
0,RECH23,Caracteristicas de la Vivienda,RECH23_2025.csv,Diccionario - RECH23.md,True
1,RECH0,Caracteristicas del hogar,RECH0_2025.csv,Diccionario - RECH0.md,True
2,RECH1,Caracteristicas del hogar,RECH1_2025.csv,Diccionario - RECH1.md,True
3,RECH4,Caracteristicas del hogar,RECH4_2025.csv,Diccionario - RECH4.md,False
4,RECHM,Caracteristicas del hogar,RECHM_2025.csv,Diccionario - RECHM.md,False
5,REC0111,Datos Básicos de MEF,REC0111_2025.csv,Diccionario - REC0111.md,True
6,REC91,Datos Básicos de MEF,REC91_2025.csv,Diccionario - REC91.md,True
7,DIT,Inmunización y Salud,DIT_2025.csv,Diccionario - DIT.md,False
8,REC42,Inmunización y Salud,REC42_2025.csv,Diccionario - REC42.md,False
9,REC43,Inmunización y Salud,REC43_2025.csv,Diccionario - REC43.md,True


• Variables seleccionadas por módulo (validadas en su diccionario):
  • REC44: ['caseid', 'child_idx', 'age_months', 'weight_raw', 'height_raw', 'haz_raw', 'waz_raw', 'whz_raw', 'hb_raw']
  • RECH23: ['hhid', 'water_source', 'sanitation', 'floor_material', 'wall_material', 'roof_material', 'wealth_quintile', 'wealth_score']
  • RECH0: ['hhid', 'household_persons', 'children_under5', 'department_code', 'residence_code', 'altitude_m']
  • RECH1: ['hhid', 'roster_idx', 'sex_code', 'age_years', 'mother_line']
  • REC0111: ['caseid', 'hhid_mother', 'mother_education', 'mother_language']
  • REC91: ['caseid', 'juntos_code']
  • REC43: ['caseid', 'child_idx', 'diarrhea_code', 'fever_code', 'cough_code', 'vitamin_a_6m_code']
  • REC95: ['caseid', 'child_idx', 'iron_syrup_code', 'iron_drops_code', 'iron_micronutrients_code']


## 1. Exploración Inicial (Diagnóstico)

Este diagnóstico registra el estado de cada módulo seleccionado inmediatamente después de su lectura y antes de los cruces, filtros de población o limpieza para el modelado. Los espacios en blanco ya se normalizan como valores nulos durante la carga. Se muestran dimensiones, nulos por variable y duplicados completos por módulo; todavía no se elimina ninguna fila.

In [12]:
# Linea base de los modulos leidos, antes de cruces, filtros o limpieza del modelado.
resumen_inicial = []
nulos_iniciales = []
for modulo, frame in sources.items():
    nulos_modulo = frame.isnull().sum()
    resumen_inicial.append({
        "modulo": modulo,
        "filas": frame.shape[0],
        "columnas": frame.shape[1],
        "nulos_totales": int(nulos_modulo.sum()),
        "filas_duplicadas_completas": int(frame.duplicated().sum()),
    })
    nulos_iniciales.append(
        nulos_modulo.rename_axis("variable").rename("nulos").reset_index().assign(modulo=modulo)
    )

print("• Diagnóstico inicial por módulo")
show_table("Dimensiones y calidad inicial por módulo", pd.DataFrame(resumen_inicial).set_index("modulo"))
print("• Revisión detallada de valores nulos")
show_table("Valores nulos iniciales por variable y módulo", pd.concat(nulos_iniciales, ignore_index=True).set_index(["modulo", "variable"]))

• Diagnóstico inicial por módulo


#### Dimensiones y calidad inicial por módulo

,filas,columnas,nulos_totales,filas_duplicadas_completas
modulo,,,,
REC44,18841,9,3864,0
RECH23,37331,8,27861,0
RECH0,37331,6,0,0
RECH1,131929,5,84538,0
REC0111,36260,4,7120,0
REC91,36260,2,36260,0
REC43,18841,6,12550,0
REC95,18841,5,2691,0


• Revisión detallada de valores nulos


#### Valores nulos iniciales por variable y módulo

nulos
modulo  variable                       
REC44   caseid                        0
        child_idx                     0
        age_months                  552
        weight_raw                  552
        height_raw                  552
        haz_raw                     552
        waz_raw                     552
        whz_raw                     552
        hb_raw                      552
RECH23  hhid                          0
        water_source               3762
        sanitation                 3762
        floor_material             4271
        wall_material              4271
        roof_material              4271
        wealth_quintile            3762
        wealth_score               3762
RECH0   hhid                          0
        household_persons             0
        children_under5               0
        department_code               0
        residence_code                0
        altitude_m                    0
RECH1   hhid                          0
        roster_idx                    0
        sex_code                      0
        age_years                     0
        mother_line               84538
REC0111 caseid                        0
        hhid_mother                   0
        mother_education           3560
        mother_language            3560
REC91   caseid                        0
        juntos_code               36260
REC43   caseid                        0
        child_idx                     0
        diarrhea_code               137
        fever_code                  137
        cough_code                  137
        vitamin_a_6m_code         12139
REC95   caseid                        0
        child_idx                     0
        iron_syrup_code             897
        iron_drops_code             897
        iron_micronutrients_code    897

## Cruce infantil, materno y del hogar

`RECH1` contiene el sexo del miembro del hogar, pero no el índice de nacimiento del archivo de antropometría. Para no asignar el sexo a un hermano incorrecto, se enlaza por hogar, línea de la madre y edad en años solo si hay exactamente un candidato. Los casos sin candidato único quedan nulos y se reportan.

In [13]:
def assert_unique(frame, keys, label):
    duplicate_count = int(frame.duplicated(keys, keep=False).sum())
    if duplicate_count:
        examples = frame.loc[frame.duplicated(keys, keep=False), keys].head(5).to_dict("records")
        raise ValueError(
            f"{label} no es único por {keys}: {duplicate_count} filas duplicadas. Ejemplos: {examples}"
        )

def merge_checked(left, right, keys, label):
    assert_unique(right, keys, label)
    before = len(left)
    indicator = f"_merge_{label}"
    merged = left.merge(
        right, how="left", on=keys, validate="many_to_one", indicator=indicator,
    )
    if len(merged) != before:
        raise AssertionError(f"El cruce con {label} multiplicó filas ({before} -> {len(merged)})")
    counts = merged[indicator].value_counts(dropna=False).to_dict()
    print(f"  • Cobertura {label}: {counts}")
    return merged.drop(columns=indicator)

child = sources["REC44"].copy()
child["caseid"] = child["caseid"].astype("string").str.strip()
child["child_idx"] = child["child_idx"].astype("string").str.strip()
CHILD_KEYS = ["caseid", "child_idx"]
assert_unique(child, CHILD_KEYS, "REC44")

rec43 = sources["REC43"].copy()
rec43["child_idx"] = rec43["child_idx"].astype("string").str.strip()
rec95 = sources["REC95"].copy()
rec95["child_idx"] = rec95["child_idx"].astype("string").str.strip()
mothers = sources["REC0111"].copy()
juntos = sources["REC91"].copy()
hh = sources["RECH0"].copy()
housing = sources["RECH23"].copy()
roster = sources["RECH1"].copy()

child = merge_checked(child, rec43, CHILD_KEYS, "REC43")
child = merge_checked(child, rec95, CHILD_KEYS, "REC95")
child = merge_checked(child, mothers, ["caseid"], "REC0111")
child = merge_checked(child, juntos, ["caseid"], "REC91")

child["hhid"] = child["caseid"].astype("string").str.split().str[0]
child["mother_line"] = child["caseid"].astype("string").str.split().str[-1]
if "hhid_mother" in child:
    mismatch = child["hhid_mother"].notna() & child["hhid"].ne(child["hhid_mother"])
    print(f"• CASEID vs HHID de la madre no coincidentes: {int(mismatch.sum())}")

for frame, label in ((hh, "RECH0"), (housing, "RECH23")):
    frame["hhid"] = frame["hhid"].astype("string").str.strip()
    assert_unique(frame, ["hhid"], label)
child = merge_checked(child, hh, ["hhid"], "RECH0")
child = merge_checked(child, housing, ["hhid"], "RECH23")

# Asignar sexo solo con una pareja hogar + línea materna + edad inequívoca.
roster["hhid"] = roster["hhid"].astype("string").str.strip()
roster["mother_line"] = roster["mother_line"].astype("string").str.strip()
roster["age_years_num"] = pd.to_numeric(roster["age_years"], errors="coerce")
roster["age_group"] = roster["age_years_num"].where(roster["age_years_num"].between(0, 4))
roster["sex_code_num"] = pd.to_numeric(roster["sex_code"], errors="coerce")
roster_key = ["hhid", "mother_line", "age_group"]
roster_candidates = roster.dropna(subset=roster_key).copy()
candidate_counts = roster_candidates.groupby(roster_key, dropna=False).size()
unique_keys = candidate_counts[candidate_counts.eq(1)].index
unique_roster = roster_candidates.set_index(roster_key).loc[unique_keys].reset_index()
unique_roster = unique_roster[roster_key + ["sex_code_num"]]

child["age_num_for_match"] = pd.to_numeric(child["age_months"], errors="coerce")
child["age_group"] = np.floor(child["age_num_for_match"] / 12)
ambiguous_keys = candidate_counts[candidate_counts.gt(1)].index
ambiguous = pd.MultiIndex.from_frame(child[roster_key]).isin(ambiguous_keys)
child = child.merge(unique_roster, how="left", on=roster_key, validate="many_to_one")
child["sex"] = child["sex_code_num"].map({1: "Hombre", 2: "Mujer"})
unmatched_sex = int(child["sex"].isna().sum())
ambiguous_sex = int(ambiguous.sum())
print(f"• Asignación de sexo: {unmatched_sex} sin candidato único; {ambiguous_sex} con candidatos ambiguos.")

# Limitar población y geografía antes del EDA solicitado.
age_for_filter = pd.to_numeric(child["age_months"], errors="coerce")
dept_for_filter = pd.to_numeric(child["department_code"], errors="coerce")
region_mask = dept_for_filter.isin(DEPARTMENT_CODES)
age_mask = age_for_filter.between(AGE_MIN_MONTHS, AGE_MAX_MONTHS, inclusive="both")
print(f"• Filas antes de filtrar edad y macrorregión: {len(child):,}")
child = child.loc[region_mask & age_mask].copy()
child["department"] = pd.to_numeric(child["department_code"], errors="coerce").map(DEPARTMENT_CODES)
print(f"• Filas después de filtrar edad y macrorregión norte: {len(child):,}")
if child.empty:
    raise ValueError("No hay niños de 6–59 meses en los ocho departamentos solicitados.")

  • Cobertura REC43: {'both': 18841, 'left_only': 0, 'right_only': 0}
  • Cobertura REC95: {'both': 18841, 'left_only': 0, 'right_only': 0}
  • Cobertura REC0111: {'both': 18841, 'left_only': 0, 'right_only': 0}
  • Cobertura REC91: {'both': 18841, 'left_only': 0, 'right_only': 0}
• CASEID vs HHID de la madre no coincidentes: 0
  • Cobertura RECH0: {'both': 18841, 'left_only': 0, 'right_only': 0}
  • Cobertura RECH23: {'both': 18841, 'left_only': 0, 'right_only': 0}
• Asignación de sexo: 877 sin candidato único; 251 con candidatos ambiguos.
• Filas antes de filtrar edad y macrorregión: 18,841
• Filas después de filtrar edad y macrorregión norte: 5,398


## 3. Detección y Tratamiento de Errores

El diagnóstico de esta sección se realiza sobre la población integrada y filtrada, antes de imputar variables, excluir casos sin etiqueta o codificar features. Los cruces ya protegen las cardinalidades mediante validaciones de unicidad.

### 3.1 Verificación de nulos y duplicados

Se revisan los nulos por columna, las filas completamente duplicadas y las llaves repetidas (`CASEID` + orden del niño). Los valores faltantes de las features se imputan en la etapa de preparación; los casos sin etiqueta de Hb se reportan y no se usan para entrenar/exportar. No se eliminan duplicados automáticamente.

### 3.2 Dimensiones antes de la limpieza

Se guarda esta forma como línea base para compararla con la matriz final después de limpiar y codificar.

In [14]:
def numeric_raw(series):
    return pd.to_numeric(series, errors="coerce")

age_for_target = numeric_raw(child["age_months"])
hb_raw_g_dl = numeric_raw(child["hb_raw"]).where(lambda s: s.between(45, 175)) / 10.0
altitude = numeric_raw(child["altitude_m"])
feet_thousands = altitude * 3.28084 / 1000.0
correction_formula = (-0.032 * feet_thousands + 0.022 * feet_thousands.pow(2)).clip(lower=0)
altitude_correction = pd.Series(
    np.where(
        altitude.isna(),
        np.nan,
        np.where(altitude.ge(ALTITUDE_MIN_METERS), correction_formula, 0),
    ),
    index=child.index,
)
hb_adjusted_audit = hb_raw_g_dl - altitude_correction
hb_cutoff = pd.Series(
    np.where(age_for_target < 24, HB_CUTOFF_6_23_MONTHS, HB_CUTOFF_24_59_MONTHS),
    index=child.index,
)
child["target_anemia_nivel"] = np.select(
    [
        hb_adjusted_audit.lt(HB_SEVERE_CUTOFF),
        hb_adjusted_audit.ge(HB_SEVERE_CUTOFF) & hb_adjusted_audit.lt(HB_MODERATE_CUTOFF),
        hb_adjusted_audit.ge(HB_MODERATE_CUTOFF) & hb_adjusted_audit.lt(hb_cutoff),
        hb_adjusted_audit.ge(hb_cutoff),
    ],
    ["anemia_severa", "anemia_moderada", "anemia_leve", "sin_anemia"],
    default=None,
)
child["target_anemia_nivel"] = pd.Series(
    child["target_anemia_nivel"], index=child.index, dtype="string"
)
child.loc[hb_adjusted_audit.isna(), "target_anemia_nivel"] = pd.NA

feature_columns = [
    "age_months", "sex", "weight_raw", "height_raw", "haz_raw", "waz_raw", "whz_raw",
    "water_source", "sanitation", "floor_material", "wall_material", "roof_material",
    "wealth_quintile", "wealth_score", "department", "residence_code", "altitude_m",
    "household_persons", "children_under5", "mother_education", "mother_language",
    "juntos_code", "diarrhea_code", "fever_code", "cough_code", "vitamin_a_6m_code",
    "iron_syrup_code", "iron_drops_code", "iron_micronutrients_code",
]
numeric_raw_columns = [
    "age_months", "weight_raw", "height_raw", "haz_raw", "waz_raw", "whz_raw",
    "wealth_score", "altitude_m", "household_persons", "children_under5",
    "department_code", "residence_code", "sex_code", "age_years",
    "diarrhea_code", "fever_code", "cough_code", "vitamin_a_6m_code",
    "iron_syrup_code", "iron_drops_code", "iron_micronutrients_code", "hb_raw",
]
before = child.copy()
dimensiones_pre_limpieza = before.shape
print(f"• Dimensiones iniciales antes de limpieza (filas, columnas): {dimensiones_pre_limpieza}")
print("\n• Calidad de datos: nulos antes de la limpieza")
show_table("Valores nulos por variable del modelo antes de limpiar", before[feature_columns + ["target_anemia_nivel"]].isnull().sum().sort_values(ascending=False).to_frame("nulos"))

conversion_failures = {}
for column in numeric_raw_columns:
    if column in before:
        raw = before[column].astype("string")
        parsed = pd.to_numeric(raw, errors="coerce")
        conversion_failures[column] = int((raw.notna() & parsed.isna()).sum())
print("\n• Calidad de datos: conversiones numéricas")
show_table("Valores no convertibles a número por variable", pd.Series(conversion_failures, name="conversiones_fallidas").to_frame())
print("\n• Tipos de datos de las variables del modelo")
show_table("Tipos de datos iniciales por variable", before[feature_columns + ["target_anemia_nivel"]].dtypes.astype(str).to_frame("dtype"))
print(f"\n• Filas completamente duplicadas: {int(before.duplicated().sum())}")
print(f"• Duplicados de llave CASEID + orden del niño: {int(before.duplicated(CHILD_KEYS).sum())}")
print("\n• Distribución del objetivo antes de la limpieza (incluye casos sin etiqueta)")
show_table("Distribución inicial de niveles de anemia", before["target_anemia_nivel"].value_counts(dropna=False).to_frame("registros"))

• Dimensiones iniciales antes de limpieza (filas, columnas): (5398, 40)

• Calidad de datos: nulos antes de la limpieza


#### Valores nulos por variable del modelo antes de limpiar

,nulos
juntos_code,5398
vitamin_a_6m_code,3151
roof_material,127
floor_material,127
wall_material,127
target_anemia_nivel,99
sex,98
iron_syrup_code,3
iron_micronutrients_code,3
iron_drops_code,3



• Calidad de datos: conversiones numéricas


#### Valores no convertibles a número por variable

,conversiones_fallidas
age_months,0
weight_raw,0
height_raw,0
haz_raw,0
waz_raw,0
whz_raw,0
wealth_score,0
altitude_m,0
household_persons,0
children_under5,0


... (2 outputs left out)

,dtype
age_months,str
sex,str
weight_raw,str
height_raw,str
haz_raw,str
waz_raw,str
whz_raw,str
water_source,str
sanitation,str
floor_material,str



• Filas completamente duplicadas: 0
• Duplicados de llave CASEID + orden del niño: 0

• Distribución del objetivo antes de la limpieza (incluye casos sin etiqueta)


#### Distribución inicial de niveles de anemia

,registros
target_anemia_nivel,
sin_anemia,4005
anemia_leve,859
anemia_moderada,431
<NA>,99
anemia_severa,4


## Limpieza, codificación, escalado y exportación

Se normalizan unidades (campos ENDES almacenados en décimos), códigos de no respuesta y tipos. Las features numéricas se imputan con mediana y se estandarizan; las categóricas se imputan con la moda y se codifican con One-Hot. El preprocesador se ajusta solo sobre el conjunto de entrenamiento para evitar fuga estadística. No se aplica sobremuestreo; se reporta el balance observado.

In [15]:
cleaned = child.copy()

def to_num(column):
    return pd.to_numeric(cleaned[column], errors="coerce") if column in cleaned else pd.Series(np.nan, index=cleaned.index)

cleaned["edad_meses"] = to_num("age_months")
cleaned["peso_kg"] = to_num("weight_raw").where(lambda s: s.between(15, 500)) / 10.0
cleaned["talla_cm"] = to_num("height_raw").where(lambda s: s.between(400, 1500)) / 10.0
for source, target in (("haz_raw", "talla_edad_z"), ("waz_raw", "peso_edad_z"), ("whz_raw", "peso_talla_z")):
    values = to_num(source).where(lambda s: s.between(-500, 500))
    values = values.mask(values.isin([9998, 9999, 99998, 99999]))
    cleaned[target] = values / 10.0
cleaned["indice_riqueza_score"] = to_num("wealth_score").where(lambda s: s.between(-100000, 100000)) / 100000.0
cleaned["altitud_m"] = to_num("altitude_m").where(lambda s: s.between(-24, 5100))
cleaned["personas_hogar"] = to_num("household_persons").where(lambda s: s.between(1, 25))
cleaned["ninos_menores_5"] = to_num("children_under5").where(lambda s: s.between(0, 24))

def clean_category(column, sentinels=("8", "9", "98", "99", "9998", "9999")):
    values = cleaned[column].astype("string").str.strip()
    return values.mask(values.isna() | values.isin(sentinels))

category_sources = {
    "fuente_agua": "water_source", "servicio_higienico": "sanitation",
    "material_piso": "floor_material", "material_paredes": "wall_material",
    "material_techo": "roof_material", "quintil_riqueza": "wealth_quintile",
    "educacion_madre": "mother_education", "lengua_madre": "mother_language",
    "juntos": "juntos_code",
}
for output, source in category_sources.items():
    cleaned[output] = clean_category(source)
cleaned["area_residencia"] = to_num("residence_code").map({1: "Urbano", 2: "Rural"})
cleaned["sexo"] = cleaned["sex"]

def yes_no(series, yes_values, no_values, unknown_values):
    values = pd.to_numeric(series, errors="coerce")
    result = pd.Series(pd.NA, index=series.index, dtype="string")
    result.loc[values.isin(yes_values)] = "Sí"
    result.loc[values.isin(no_values)] = "No"
    result.loc[values.isin(unknown_values)] = pd.NA
    return result

cleaned["diarrea_14d"] = yes_no(cleaned["diarrhea_code"], {1, 2}, {0}, {8, 9, 98})
cleaned["fiebre_14d"] = yes_no(cleaned["fever_code"], {1}, {0}, {8, 9, 98})
cleaned["tos_14d"] = yes_no(cleaned["cough_code"], {1, 2}, {0}, {8, 9, 98})
cleaned["vitamina_a_6m"] = yes_no(cleaned["vitamin_a_6m_code"], {1}, {0}, {8, 9, 98})

iron_codes = cleaned[["iron_syrup_code", "iron_drops_code", "iron_micronutrients_code"]].apply(
    lambda s: pd.to_numeric(s, errors="coerce")
)
iron_yes = iron_codes.eq(1).any(axis=1)
iron_no = iron_codes.notna().all(axis=1) & iron_codes.eq(2).all(axis=1)
cleaned["hierro_12m"] = pd.Series(pd.NA, index=cleaned.index, dtype="string")
cleaned.loc[iron_yes, "hierro_12m"] = "Sí"
cleaned.loc[iron_no, "hierro_12m"] = "No"

model_numeric = [
    "edad_meses", "peso_kg", "talla_cm", "talla_edad_z", "peso_edad_z",
    "peso_talla_z", "indice_riqueza_score", "altitud_m", "personas_hogar",
    "ninos_menores_5",
]
model_categorical = [
    "sexo", "fuente_agua", "servicio_higienico", "material_piso", "material_paredes",
    "material_techo", "quintil_riqueza", "department", "area_residencia",
    "educacion_madre", "lengua_madre", "juntos", "diarrea_14d", "fiebre_14d",
    "tos_14d", "vitamina_a_6m", "hierro_12m",
]
model_features = model_numeric + model_categorical

labeled = cleaned.loc[cleaned["target_anemia_nivel"].notna()].copy()
if labeled.empty:
    raise ValueError("No hay Hb infantil válida para construir target; revise HW53 y sus códigos.")
y = labeled["target_anemia_nivel"].astype("string")
X = labeled[model_features].copy()
missing_feature_columns = [c for c in model_features if c not in X]
if missing_feature_columns:
    raise ValueError(f"Faltan features esperadas después de consolidar: {missing_feature_columns}")

# Reemplazar infinitos antes de imputar y mantener los faltantes categóricos como NA.
X[model_numeric] = X[model_numeric].replace([np.inf, -np.inf], np.nan)
for column in model_categorical:
    X[column] = X[column].astype(object).where(X[column].notna(), np.nan)
all_missing_categories = [c for c in model_categorical if X[c].isna().all()]
if all_missing_categories:
    print(
        "• Aviso: estas variables no tienen respuestas observadas en los registros filtrados "
        f"y se representan como 'Sin dato': {all_missing_categories}"
    )
    for column in all_missing_categories:
        X[column] = "Sin dato"


• Aviso: estas variables no tienen respuestas observadas en los registros filtrados y se representan como 'Sin dato': ['juntos']


## 5. División estratificada del dataset

Se separan las observaciones en entrenamiento (80 %) y prueba (20 %) antes de ajustar imputadores, escaladores o codificadores. El conjunto de prueba queda reservado para evaluar el modelo; el preprocesador se ajusta solo con entrenamiento y luego transforma ambos conjuntos, evitando fuga de información. La estratificación por nivel de anemia es una práctica habitual en clasificación para mantener la representación de las clases, como se ilustra en estudios de predicción de anemia con un conjunto de prueba independiente ([Kamalzadeh et al., 2025; estudio en adultos, split 70/30](https://pubmed.ncbi.nlm.nih.gov/41413508/); [documentación de `train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)).

**Límite de los datos:** hay solo 4 registros de anemia severa. Como los recuentos deben ser enteros, con esa clase no es posible mantener exactamente la misma proporción en ambas particiones al asignar el 20 % a prueba; `stratify=y` conserva la distribución de la forma más cercana posible. La tabla siguiente permite comprobar los recuentos, porcentajes y diferencias observadas.

In [16]:
# Division reproducible 80/20 con estratificacion obligatoria por nivel de anemia.
class_counts = y.value_counts()
if len(class_counts) < 2:
    raise ValueError(f"Se requieren al menos dos clases para una division estratificada; distribucion={class_counts.to_dict()}")
if class_counts.min() < 2:
    raise ValueError(
        "No se puede estratificar: cada clase debe tener al menos dos observaciones. "
        f"Distribucion={class_counts.to_dict()}"
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("• Dimensiones de las particiones de entrenamiento y prueba:")
print(f"  • X_train: {X_train.shape}")
print(f"  • X_test:  {X_test.shape}")
print(f"  • y_train: {y_train.shape}")
print(f"  • y_test:  {y_test.shape}")

conteos_train = y_train.value_counts().reindex(TARGET_ORDER, fill_value=0)
conteos_test = y_test.value_counts().reindex(TARGET_ORDER, fill_value=0)
distribucion_clases = pd.DataFrame({
    "entrenamiento_n": conteos_train,
    "entrenamiento_pct": conteos_train.div(len(y_train)).mul(100),
    "prueba_n": conteos_test,
    "prueba_pct": conteos_test.div(len(y_test)).mul(100),
})
distribucion_clases["diferencia_pct_puntos"] = (
    distribucion_clases["prueba_pct"] - distribucion_clases["entrenamiento_pct"]
)
show_table("Recuento y porcentaje de niveles de anemia por partición", distribucion_clases.round(2))

• Dimensiones de las particiones de entrenamiento y prueba:
  • X_train: (4239, 27)
  • X_test:  (1060, 27)
  • y_train: (4239,)
  • y_test:  (1060,)


#### Recuento y porcentaje de niveles de anemia por partición

,entrenamiento_n,entrenamiento_pct,prueba_n,prueba_pct,diferencia_pct_puntos
target_anemia_nivel,,,,,
sin_anemia,3204,75.58,801,75.57,-0.02
anemia_leve,687,16.21,172,16.23,0.02
anemia_moderada,345,8.14,86,8.11,-0.03
anemia_severa,3,0.07,1,0.09,0.02


In [17]:
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # Compatibilidad con scikit-learn anterior a 1.2
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot),
])
preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, model_numeric),
        ("cat", categorical_pipeline, model_categorical),
    ],
    remainder="drop",
    sparse_threshold=0,
)
preprocessor.fit(X_train, y_train)
X_train_ready = preprocessor.transform(X_train)
X_test_ready = preprocessor.transform(X_test)
X_ready = preprocessor.transform(X)
feature_names = preprocessor.get_feature_names_out()
X_ready = np.asarray(X_ready, dtype=np.float32)
print("\n• Preprocesamiento ajustado solo con entrenamiento:")
print(f"  • X_train_ready: {X_train_ready.shape}")
print(f"  • X_test_ready: {X_test_ready.shape}")
if not np.isfinite(X_ready).all():
    raise ValueError("La matriz transformada contiene NaN o infinitos.")

target_codes = pd.Categorical(y, categories=TARGET_ORDER, ordered=True).codes
if (target_codes < 0).any():
    raise ValueError(f"El target contiene categorías fuera de TARGET_ORDER: {sorted(set(y) - set(TARGET_ORDER))}")
final = pd.DataFrame(X_ready, columns=feature_names, index=X.index)
final["target_anemia_nivel"] = y.to_numpy()
final["target_anemia_codigo"] = target_codes.astype("int8")


• Preprocesamiento ajustado solo con entrenamiento:
  • X_train_ready: (4239, 91)
  • X_test_ready: (1060, 91)


## 4. Validación Post-Limpieza

Antes de exportar, se vuelven a calcular los nulos y duplicados de la matriz final y se comparan sus filas y columnas con la línea base anterior a la limpieza. Los duplicados completos en la matriz codificada se reportan, pero no se borran: dos observaciones distintas pueden tener exactamente las mismas features y target.

In [18]:
# Repetir las comprobaciones sobre la matriz que se exportara.
nulos_finales_por_columna = final.isnull().sum()
print("• Validación de valores faltantes en la matriz final")
show_table("Valores nulos por columna en la matriz final", nulos_finales_por_columna.to_frame("nulos"))
total_nulos_finales = int(nulos_finales_por_columna.sum())
print(f"  • Total de valores nulos finales: {total_nulos_finales}")
if total_nulos_finales:
    raise ValueError("La matriz final todavia contiene valores nulos.")

duplicados_finales = int(final.duplicated().sum())
print(f"• Filas completamente duplicadas en la matriz final: {duplicados_finales}")

comparativa_dimensiones = pd.DataFrame(
    [dimensiones_pre_limpieza, final.shape],
    index=["Inicial (antes de limpieza)", "Final (features + target)"],
    columns=["filas", "columnas"],
)
print("\n• Comparativa de dimensiones iniciales y finales")
show_table("Dimensiones antes y después de la limpieza", comparativa_dimensiones)
print(
    "  • Diferencia final - inicial: "
    f"{final.shape[0] - dimensiones_pre_limpieza[0]:+,} filas; "
    f"{final.shape[1] - dimensiones_pre_limpieza[1]:+,} columnas."
)
print(f"  • Filas excluidas por falta de etiqueta de hemoglobina: {len(cleaned) - len(labeled)}")
print("\n• Balance final de clases")
show_table("Recuento final de registros por nivel de anemia", final["target_anemia_nivel"].value_counts().reindex(TARGET_ORDER, fill_value=0).to_frame("registros"))
print(f"  • Codificación ordinal del objetivo: {dict(enumerate(TARGET_ORDER))}")
print(f"  • Columnas de salida: {len(final.columns)} (sin hemoglobina cruda ni ajustada)")

final.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n• Archivo CSV exportado: {OUTPUT_CSV}")

• Validación de valores faltantes en la matriz final


#### Valores nulos por columna en la matriz final

,nulos
num__edad_meses,0
num__peso_kg,0
num__talla_cm,0
num__talla_edad_z,0
num__peso_edad_z,0
...,...
cat__vitamina_a_6m_Sí,0
cat__hierro_12m_No,0
cat__hierro_12m_Sí,0
target_anemia_nivel,0


  • Total de valores nulos finales: 0
• Filas completamente duplicadas en la matriz final: 0

• Comparativa de dimensiones iniciales y finales


#### Dimensiones antes y después de la limpieza

,filas,columnas
Inicial (antes de limpieza),5398,40
Final (features + target),5299,93


  • Diferencia final - inicial: -99 filas; +53 columnas.
  • Filas excluidas por falta de etiqueta de hemoglobina: 99

• Balance final de clases


#### Recuento final de registros por nivel de anemia

,registros
target_anemia_nivel,
sin_anemia,4005
anemia_leve,859
anemia_moderada,431
anemia_severa,4


  • Codificación ordinal del objetivo: {0: 'sin_anemia', 1: 'anemia_leve', 2: 'anemia_moderada', 3: 'anemia_severa'}
  • Columnas de salida: 93 (sin hemoglobina cruda ni ajustada)

• Archivo CSV exportado: C:\PROYECTODSI\ENDES_2025_MLP_macrorregion_norte.csv
